In [ ]:
# Colab setup (added for Google Colab): run this cell first. It does nothing outside Colab.
# It fetches this practical's data/ folder from GitHub, so the notebook runs unchanged.
import os, shutil, subprocess, sys
from pathlib import Path

REPO = "https://github.com/TAHMO/NOAA.git"
BRANCH = "zambia-workshop-2"
FOLDER = "Exercise_3/Practical_1_weather_yield"

if ("google.colab" in sys.modules or "COLAB_RELEASE_TAG" in os.environ) and not Path("data").is_dir():
    clone = Path("/content/NOAA")
    if not (clone / FOLDER / "data").is_dir():
        shutil.rmtree(clone, ignore_errors=True)
        subprocess.run(["git", "clone", "-q", "--depth", "1", "--filter=blob:none", "--sparse",
                        "-b", BRANCH, REPO, str(clone)], check=True)
        subprocess.run(["git", "-C", str(clone), "sparse-checkout", "set", FOLDER + "/data"], check=True)
    os.chdir(clone / FOLDER)
    print("Data ready:", Path("data").resolve())


# From weather to agricultural risk
## Notebook toolkit

This practical explores how **weather** and **seasonal forecasts** relate to national **maize yield** in six
African countries, 1984–2024. The aim is to see when weather and forecasts can warn of a higher risk of a
poor harvest, and how strong that warning really is.

### How to use this notebook

1. **Run the setup cell** in Section 1 once.
2. **Choose the country once** in Section 1. Every section uses that choice. To switch country, change it there
   and use **Run → Run All Cells**.
3. Each section then follows the same pattern:
   - 🎯 an **objective** box: what the section is for and what you will learn;
   - a **settings** cell (where there is something to change, such as an example year);
   - the **chart** and **tables** (the code is folded away: click the grey bar to see it);
   - a **key result** box that reads the main numbers back in plain words;
   - 💬 **questions to discuss**, the same as in the Word task sheet.
4. **After changing a setting, run every cell below it in that subsection** (chart, table and key result),
   so the numbers in the key-result box match the chart.
5. Section 8 brings together **what we learned**.
6. **Section 9 is your group output:** a country brief with your own interpretation, which you report back to the room.

Work through **all nine sections** in order: Tasks 1–7, then what we learned, then your country brief.

| Section | What you will learn | Task |
|---|---|---|
| 1 | What data you have for your country | Task 1 |
| 2 | How yield and weather vary, and how a *local rank* removes the long-term trend | Task 2 |
| 3 | Whether observed weather and seasonal forecasts separate good and poor harvests | Task 3 |
| 4 | Two more weather measures: hot days and SPI (a drought index) | Task 4 |
| 5 | What happens *within* a season, month by month | Task 5 |
| 6 | How El Niño and La Niña relate to yield | Task 6 |
| 7 | How good the seasonal forecasts are | Task 7 |
| 8 | What we learned | — |
| 9 | **Your country brief:** your own interpretation and advice, reported back | Group output · everyone |

### 1. Setup and datasets

> 🎯 **Objective:** get the notebook running and get to know the data for your country.
>
> **You will learn to:**
> - load the yield, weather and forecast data for one country
> - read the basic numbers: how many years, typical yield, rainfall and temperature
>
> **Task sheet:** Task 1

**In short:** Run the setup cell below once. It loads the tools; it shows nothing except a short confirmation. Python packages needed: numpy, pandas, matplotlib, jinja2 and jupyter.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, HTML

DATA_DIR = Path("data")
plt.rcParams.update({"figure.figsize": (10, 4), "font.size": 11})

labels = {"seasonal_rainfall_mm": "Seasonal rainfall (mm)", "mean_spi3": "Seasonal mean SPI-3", "mean_tmax_c": "Mean daily maximum temperature (°C)", "hot_days_gt30c": "Country-average days >30°C", "hot_days_gt32c": "Country-average days >32°C", "hot_days_gt35c": "Country-average days >35°C"}
forecast_metrics = [
    ("rainfall_mm", "seasonal_rainfall_mm", "Seasonal rainfall (mm)", "TAMSAT"),
    ("mean_tmax_c", "mean_tmax_c", "Mean daily maximum temperature (°C)", "ERA5"),
]

def five_year_ranks(years, yields):
    years = np.asarray(years)
    values = pd.Series(np.asarray(yields, dtype=float))
    if len(years) < 5 or not np.all(np.diff(years) == 1):
        raise ValueError("This core example needs at least five consecutive harvest years; do not silently bridge gaps.")
    ranks = []
    for i in range(len(values)):
        start = min(max(i - 2, 0), len(values) - 5)
        window = values.iloc[start:start + 5]
        ranks.append(window.rank(method="average").iloc[i - start] if window.notna().all() else np.nan)
    return np.asarray(ranks)


def load_country(country):
    path = DATA_DIR / f"{country.lower()}.csv"
    if not path.exists():
        raise FileNotFoundError(f"Cannot find {path}. Open the notebook from its own folder, or update DATA_DIR.")
    data = pd.read_csv(path).sort_values("harvest_year").reset_index(drop=True)
    months = pd.read_csv(DATA_DIR / f"{country.lower()}_monthly_weather.csv", parse_dates=["month"])
    assert data.country.eq(country).all() and data.harvest_year.is_unique
    return data, months

def load_forecasts(country, data):
    forecast = pd.read_csv(DATA_DIR / f"{country.lower()}_seasonal_forecasts_lead0.csv")
    forecast = forecast.sort_values("harvest_year").reset_index(drop=True)
    comparison = forecast.merge(data[["harvest_year", "seasonal_rainfall_mm", "mean_tmax_c"]],
                                on="harvest_year", validate="one_to_one")
    return forecast, comparison

def yield_groups(df):
    q1, q2 = df.yield_t_ha.quantile([1/3, 2/3])
    if not q1 < q2:
        raise ValueError("Yield terciles coincide: choose explicit category boundaries for this record.")
    category_order = ["Below", "Average", "Above"]
    df["absolute_yield_category"] = pd.cut(df.yield_t_ha, [-np.inf, q1, q2, np.inf], labels=category_order)
    category_definitions = pd.DataFrame({"category": category_order,
        "yield_t_ha": [f"≤ {q1:.3f}", f"> {q1:.3f} to ≤ {q2:.3f}", f"> {q2:.3f}"],
        "years": [int(df.absolute_yield_category.eq(c).sum()) for c in category_order]})
    rank_order = sorted(df.local_yield_rank_5.dropna().unique())
    return category_order, rank_order, category_definitions

def weather_boxes(ax, df, metric, grouping, order):
    groups = [df.loc[df[grouping].eq(g), metric].dropna().to_numpy() for g in order]
    tick_text = [f"{g:g}" if isinstance(g, (float, np.floating)) else str(g) for g in order]
    bp = ax.boxplot(groups, patch_artist=True, widths=.6,
                    medianprops={"color": "#222222", "linewidth": 1.6},
                    flierprops={"marker": "o", "markersize": 4, "alpha": .65})
    for box in bp["boxes"]: box.set_facecolor("#b9d5e5")
    ax.set_xticks(range(1, len(order)+1), [str(t) + chr(10) + f"(n={len(v)})" for t, v in zip(tick_text, groups)])
    ax.set_ylabel(labels[metric]); ax.grid(axis="y", alpha=.2); ax.set_axisbelow(True)
    ax.set_xlabel("Local yield rank" if grouping == "local_yield_rank_5" else "Absolute yield category (terciles)")



COUNTRIES = ["Zambia", "Malawi", "Zimbabwe", "Botswana", "Mozambique", "Nigeria"]


def season_text(country):
    return "May–September" if country == "Nigeria" else "November–April"


def num(x, digits=2):
    """Format a number, or a dash when it is missing."""
    return "—" if x is None or not np.isfinite(x) else f"{x:.{digits}f}"


def strength(r):
    """Plain-language label for a correlation (a teaching convention: 0.2 / 0.4 / 0.6 cut-offs)."""
    if r is None or not np.isfinite(r):
        return "undefined"
    a = abs(r)
    if a < .2:
        return "very weak"
    word = "strong" if a >= .6 else "moderate" if a >= .4 else "weak"
    return f"{word} {'positive' if r > 0 else 'negative'}"


def key_result(title, lines, colour="#367a9c"):
    """A coloured box that reads the main result back in plain words."""
    items = "".join(f"<li style='margin:4px 0'>{line}</li>" for line in lines)
    display(HTML(f"<div style='border-left:6px solid {colour};background:#f4f8fa;color:#1d2b33;"
                 f"padding:10px 16px;margin:14px 0 8px 0;border-radius:4px;font-size:14px'>"
                 f"<b style='font-size:15px'>{title}</b><ul style='margin:6px 0 0 0;padding-left:20px'>"
                 f"{items}</ul></div>"))


def results_table(table, caption, formats=None, highlight=None, gradient=None, colours=None):
    """A clean, coloured table. gradient = {column or tuple of columns: colormap or (colormap, vmin, vmax)}; colours = {column: {value: colour}};
    highlight = boolean array of rows to mark."""
    table = table.reset_index(drop=True)
    st = table.style.hide(axis="index").set_caption(caption)
    st = st.format(formats or {}, na_rep="—")
    st = st.set_table_styles([
        {"selector": "caption", "props": "caption-side:top;text-align:left;font-weight:bold;"
                                         "font-size:14px;color:inherit;padding:6px 0"},
        {"selector": "th", "props": "background:#2f5d73;color:white;padding:6px 10px;text-align:left"},
        {"selector": "td", "props": "padding:5px 10px;border-bottom:1px solid #dde5ea;text-align:left;"
                                    ""},
        {"selector": "", "props": "border-collapse:collapse;margin:8px 0 14px 0;background:white;color:#1d2b33"}])
    for columns, spec in (gradient or {}).items():
        cmap, vmin, vmax = spec if isinstance(spec, tuple) else (spec, None, None)
        columns = list(columns) if isinstance(columns, tuple) else [columns]
        if vmin is None and cmap in ("coolwarm", "BrBG", "PuOr", "RdBu"):   # diverging: centre on zero, or on the mean
            values = table[columns].to_numpy(dtype=float)
            centre = 0 if np.nanmin(values) < 0 < np.nanmax(values) else np.nanmean(values)
            v = np.nanmax(np.abs(values - centre)) or 1
            vmin, vmax = centre - v, centre + v
        st = st.background_gradient(cmap=cmap, subset=columns, axis=None, vmin=vmin, vmax=vmax,
                                    low=.15, high=.15)
    for column, mapping in (colours or {}).items():
        st = st.map(lambda v, m=mapping: f"background:{m[v]}" if v in m else "", subset=[column])
    if highlight is not None:
        mark = list(highlight)
        st = st.apply(lambda row: ["background:#fff3c4;font-weight:bold" if mark[row.name] else ""
                                   for _ in row], axis=1)
    display(st)


print("Setup complete. Now choose the country in the next cell.")

#### Choose the country

Change the name below and run the cell. **Every section uses this choice.** To switch country later,
change it here and use **Run → Run All Cells**.

In [ ]:
# ▶ Choose the country ONCE. Every section uses it.
COUNTRY = "Zimbabwe"  # Zambia, Malawi, Zimbabwe, Botswana, Mozambique, Nigeria

#### Your data at a glance

In [ ]:
if COUNTRY not in COUNTRIES:
    raise ValueError(f"Choose one of: {', '.join(COUNTRIES)}")
df, monthly = load_country(COUNTRY)
metrics = ["yield_t_ha", "local_yield_rank_5", "seasonal_rainfall_mm", "mean_spi3", "mean_tmax_c",
           "hot_days_gt30c", "hot_days_gt32c", "hot_days_gt35c"]
names = {**labels, "yield_t_ha": "Maize yield (t/ha)", "local_yield_rank_5": "Local yield rank (1–5)"}
overview = df[metrics].agg(["count", "min", "mean", "max"]).T
overview.insert(0, "Measure", [names[m] for m in overview.index])
overview.columns = ["Measure", "Years with data", "Lowest", "Average", "Highest"]
results_table(overview, f"{COUNTRY}: the data at a glance",
              formats={"Years with data": "{:.0f}", "Lowest": "{:.2f}", "Average": "{:.2f}", "Highest": "{:.2f}"})
low, high = df.loc[df.yield_t_ha.idxmin()], df.loc[df.yield_t_ha.idxmax()]
key_result(f"{COUNTRY} at a glance", [
    f"<b>{len(df)} harvest years</b>, {df.harvest_year.min()}–{df.harvest_year.max()}; growing season {season_text(COUNTRY)}.",
    f"Average maize yield <b>{df.yield_t_ha.mean():.2f} t/ha</b>: lowest {low.yield_t_ha:.2f} t/ha ({low.harvest_year:.0f}), "
    f"highest {high.yield_t_ha:.2f} t/ha ({high.harvest_year:.0f}).",
    f"Average seasonal rainfall <b>{df.seasonal_rainfall_mm.mean():.0f} mm</b>; average daily maximum temperature "
    f"<b>{df.mean_tmax_c.mean():.1f} °C</b>."])

# Uncomment to see data counts and yield flags
# display(df.groupby("yield_flag", dropna=False).size().rename("years").to_frame())
# display(df[["harvest_year", "expected_days", "rainfall_valid_days", "temperature_valid_days", "heat_valid_days", "spi3_valid_months", "spi3_expected_months"]])

<details><summary><b>About the datasets</b> (click to open)</summary>

- **Yield:** FAOSTAT national annual maize yield, tonnes per hectare. A = official, E = estimated, I = imputed, X = external organisation. Flags describe provenance, not quantified uncertainty.
- **Rainfall:** whole-country TAMSAT seasonal total, mm.
- **Temperature:** ERA5 seasonal mean daily maximum temperature, °C
- **Hot days:** counts above 30, 32 or 35°C at each grid cell, then country-area averaging. Fractional values are expected. These are generic thresholds, not universal crop-damage thresholds.
- **Mean SPI-3:** arithmetic average of monthly SPI-3 across the season. Monthly SPI-3 uses trailing three-month country rainfall, fitted separately for each ending month against 1991–2020. It is not SPI-6 and the seasonal mean is not itself standard normal.

Nigeria uses May–September of the harvest year; the other countries use November of the previous year–April of the harvest year.

</details>

### 2. Weather and yield

> 🎯 **Objective:** see how maize yield and weather vary from year to year, and how closely they move together.
>
> **You will learn to:**
> - read a yield time series and spot a long-term trend
> - use a **local rank** to find poor years even when yields are trending up or down
> - measure how closely a weather variable and yield move together (correlation, *r*)
> - find years that don't fit the pattern, and think about why
>
> **Task sheet:** Task 2

**In short:** Yields change over decades (better seeds, more fertiliser, policy), so a yield that was good in 1990 may be poor in 2020. A **local rank** compares each harvest only with its neighbours, which takes most of that trend out.

#### A. Local five-year yield ranks

Each harvest gets a rank from **1 (lowest) to 5 (highest)** compared with the two harvests before and the two after.

<details><summary><b>More detail</b> (click to open)</summary>

A local rank compares each yield with five nearby harvests: normally two before, the focal year, and two after. **1 = lowest; 5 = highest.** At either end of the 1984–2024 record, shift the window to retain five years. Ties get average ranks.

This reduces but does not eliminate long term trends in yield. Rank 1 is a locally low yield, not necessarily a crop failure. It has no percentage-loss interpretation.

</details>

> 💡 **Why use the rank?** A raw yield mixes two things: slow change over decades (seeds, fertiliser, policy, the economy) and the ups and downs from one year to the next. Weather acts mainly on the ups and downs. The rank keeps those and drops most of the slow change. In Section 2B, watch what happens to *r* when you move from the left panel to the right one.

In [ ]:
# Settings — Section 2A
EXAMPLE_YEAR = 2016  # harvest year to look at, 1984–2024

In [ ]:
df, monthly = load_country(COUNTRY)
if EXAMPLE_YEAR not in df.harvest_year.values:
    raise ValueError(f"Choose a harvest year from {df.harvest_year.min()} to {df.harvest_year.max()}.")
computed_ranks = five_year_ranks(df.harvest_year, df.yield_t_ha)
assert np.allclose(computed_ranks, df.local_yield_rank_5, equal_nan=True)
focal = df.loc[df.harvest_year == EXAMPLE_YEAR].iloc[0]
window = df[df.harvest_year.between(focal.rank_window_start_year, focal.rank_window_end_year)][["harvest_year", "yield_t_ha"]].copy()
window["rank"] = window.yield_t_ha.rank(method="average")
shown = window.rename(columns={"harvest_year": "Harvest year", "yield_t_ha": "Yield (t/ha)",
                               "rank": "Rank in this window (1 = lowest)"})
results_table(shown, f"{COUNTRY}: the five harvests used to rank {EXAMPLE_YEAR} (highlighted)",
              formats={"Yield (t/ha)": "{:.2f}", "Rank in this window (1 = lowest)": "{:g}"},
              highlight=window.harvest_year.eq(EXAMPLE_YEAR).to_numpy(),
              gradient={"Rank in this window (1 = lowest)": ("RdYlGn", 1, 5)})

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(11, 6), sharex=True)
mu = df.yield_t_ha.mean()
axes[0].bar(df.harvest_year, df.yield_t_ha-mu, bottom=mu, color="#657985")
axes[0].axhline(mu, color="black", lw=1)
axes[0].set_ylabel("Yield (tonnes/ha)")
axes[0].set_title(f"{COUNTRY}: yield relative to its 1984–2024 mean")
axes[1].plot(df.harvest_year, df.local_yield_rank_5, "o-", color="#333333", ms=4)
axes[1].set_yticks(range(1, 6)); axes[1].set_ylabel("Local yield rank"); axes[1].set_xlabel("Harvest year")
for ax in axes: ax.grid(alpha=.2); ax.set_axisbelow(True)
plt.tight_layout(); plt.show()

In [ ]:
slope = np.polyfit(df.harvest_year, df.yield_t_ha, 1)[0] * 10
first, last = df.head(10), df.tail(10)
rank_one = df.loc[df.local_yield_rank_5.eq(1), "harvest_year"].astype(int).tolist()
early_low = [y for y in rank_one if y < 2004]; late_low = [y for y in rank_one if y >= 2004]
key_result("Key result: yield over time", [
    f"Straight-line trend: yield {'rose' if slope > 0 else 'fell'} by about <b>{abs(slope):.2f} t/ha per decade</b>.",
    f"The first ten harvests ({first.harvest_year.min()}–{first.harvest_year.max()}) averaged <b>{first.yield_t_ha.mean():.2f} t/ha</b>; "
    f"the last ten ({last.harvest_year.min()}–{last.harvest_year.max()}) averaged <b>{last.yield_t_ha.mean():.2f} t/ha</b>.",
    f"<b>{EXAMPLE_YEAR}</b>: {focal.yield_t_ha:.2f} t/ha, local rank <b>{focal.local_yield_rank_5:g} of 5</b>.",
    f"Locally lowest harvests (rank 1): {len(early_low)} before 2004 ({', '.join(map(str, early_low)) or 'none'}) and "
    f"{len(late_low)} from 2004 ({', '.join(map(str, late_low)) or 'none'}). Local ranks find poor years in every part of the record."])

> 💬 **Questions to discuss — Task 2.1**
>
> 1. Is there a long-term trend in yield?
> 2. Pick a low-ranked and a high-ranked year from both the early and late parts of the record. How do their absolute yields compare?
> 3. What does local ranking help us identify?

#### B. Weather and yield scatter plots

Each dot is one harvest year. The left panel uses raw yield, the right panel the local rank. **r** (Pearson correlation) runs from −1 to +1: near 0 means the two don't move together; near ±1 means they move closely together.

<details><summary><b>More detail</b> (click to open)</summary>

The panels show raw yield and local ranks against the selected weather metric. Pearson correlation is a descriptive summary, not forecast skill or evidence of causation. Adjacent yield ranks use overlapping windows, so their independence cannot be assumed.

</details>

> 💡 **Hint:** try METRIC = mean_tmax_c as well as rainfall. If the raw-yield panel seems to say that hotter seasons gave *better* harvests, ask what else has gone up over the same 40 years. Which panel would you trust to show the effect of the weather?

In [ ]:
# Settings — Section 2B
METRIC = "seasonal_rainfall_mm"  # also: mean_tmax_c, mean_spi3, hot_days_gt30c, hot_days_gt32c, hot_days_gt35c

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for ax, response, label in zip(axes, ["yield_t_ha", "local_yield_rank_5"], ["Yield (tonnes/ha)", "Local yield rank"]):
    pairs = df[[METRIC, response]].dropna()
    ax.scatter(pairs[METRIC], pairs[response], s=35, alpha=.8)
    ax.set_xlabel(labels[METRIC]); ax.set_ylabel(label); ax.grid(alpha=.2)
    ax.set_title(f"r = {pairs[METRIC].corr(pairs[response]):.2f}; {len(pairs)} years")
axes[1].set_yticks(range(1, 6))
fig.suptitle(COUNTRY); plt.tight_layout(); plt.show()
scatter_metric = METRIC

In [ ]:
r_raw = df[[scatter_metric, "yield_t_ha"]].dropna().corr().iloc[0, 1]
r_rank = df[[scatter_metric, "local_yield_rank_5"]].dropna().corr().iloc[0, 1]
change = ("stronger" if abs(r_rank) > abs(r_raw) + .05 else
          "weaker" if abs(r_rank) < abs(r_raw) - .05 else "about the same")
key_result(f"Key result: {labels[scatter_metric].lower()} and yield", [
    f"With raw yield: <b>r = {r_raw:.2f}</b> ({strength(r_raw)}).",
    f"With local ranks: <b>r = {r_rank:.2f}</b> ({strength(r_rank)}). The relationship is <b>{change}</b> once most of the trend is removed.",
    f"r² = {r_rank ** 2:.2f}: a straight-line relationship with this measure matches about <b>{100 * r_rank ** 2:.0f} %</b> of the "
    f"year-to-year variation in local rank. The rest is not captured by this simple relationship: rain timing, pests, prices, "
    f"inputs and data quality can all play a part."])

#### C. Weather by yield group

The same question the other way round: in poor and good harvest years, what was the weather like? Left: years grouped by local rank. Right: years grouped into the lowest, middle and highest third of yields.

<details><summary><b>More detail</b> (click to open)</summary>

Weather distributions are grouped by **local yield rank** and by **absolute yield category**. Below, average and above are the lower, middle and upper thirds (terciles) of the country's 1984–2024 raw yield record. “Average” means the middle category, not exactly the arithmetic mean. Cut-offs are shown in tonnes/ha. Tied yields stay together, so category sizes may differ.

Boxes show the middle 50% of weather values and the median; whiskers extend to the most extreme observations within 1.5 times the interquartile range. Dots beyond the whiskers are individual observations, not automatically errors. Sample counts are shown. Fractional local ranks caused by tied yields remain separate groups; no years are rounded or dropped.

</details>

In [ ]:
# Settings — Section 2C
METRIC = "seasonal_rainfall_mm"  # also: mean_tmax_c, mean_spi3, hot_days_gt30c, hot_days_gt32c, hot_days_gt35c

In [ ]:
df, monthly = load_country(COUNTRY)
category_order, rank_order, category_definitions = yield_groups(df)
shown = category_definitions.rename(columns={"category": "Yield group", "yield_t_ha": "Yield range (t/ha)", "years": "Years"})
results_table(shown, f"{COUNTRY}: how the three yield groups are defined",
              colours={"Yield group": {"Below": "#f6d5c9", "Average": "#eeeeee", "Above": "#d4e8d4"}})

In [ ]:
category_order, rank_order, category_definitions = yield_groups(df)
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5), sharey=True)
weather_boxes(axes[0], df, METRIC, "local_yield_rank_5", rank_order)
weather_boxes(axes[1], df, METRIC, "absolute_yield_category", category_order)
axes[0].set_title("Grouped by local rank"); axes[1].set_title("Grouped by absolute yield")
fig.suptitle(f"{COUNTRY}: {labels[METRIC]}"); plt.tight_layout(); plt.show()
box_metric = METRIC

In [ ]:
median = lambda column, group: df.loc[df[column].eq(group), box_metric].median()
unit = labels[box_metric].lower()
key_result(f"Key result: {unit} in poor and good years", [
    f"Locally lowest years (rank 1): median <b>{num(median('local_yield_rank_5', 1), 1)}</b>; "
    f"locally highest years (rank 5): median <b>{num(median('local_yield_rank_5', 5), 1)}</b>.",
    f"Lowest third of yields: median <b>{num(median('absolute_yield_category', 'Below'), 1)}</b>; "
    f"highest third: median <b>{num(median('absolute_yield_category', 'Above'), 1)}</b>.",
    "Look at how much the boxes overlap: a big difference between medians with little overlap is a clearer signal than a small difference with a lot of overlap."])

#### Year by year

Use this table to find the years the questions ask about.

> 💡 **For question 1 (the year that doesn't fit):** look for local rank **1 or 2** where rainfall was **not more than about 10 % below average** and the season was **not hotter than usual**. A wetter or cooler season does not explain a poor harvest, so those years count too. Then set DETAIL_YEAR to that year in Section 5: was one month dry or hot inside a normal-looking total?

Colours: green = higher rank or wetter than average; red = hotter than average.

In [ ]:
years_table = df[["harvest_year", "yield_t_ha", "local_yield_rank_5", "seasonal_rainfall_mm", "mean_tmax_c", "hot_days_gt30c"]].copy()
rain_avg, tmax_avg = df.seasonal_rainfall_mm.mean(), df.mean_tmax_c.mean()
years_table.insert(4, "rain_vs_avg", (years_table.seasonal_rainfall_mm / rain_avg - 1) * 100)
years_table.insert(6, "tmax_vs_avg", years_table.mean_tmax_c - tmax_avg)
years_table.columns = ["Harvest year", "Yield (t/ha)", "Local rank", "Rainfall (mm)", "Rain vs average (%)",
                       "Max temp (°C)", "Max temp vs average (°C)", "Days >30 °C"]
results_table(years_table, f"{COUNTRY}: every harvest year (1984–2024 averages: rainfall {rain_avg:.0f} mm, max temp {tmax_avg:.1f} °C)",
              formats={"Yield (t/ha)": "{:.2f}", "Local rank": "{:g}", "Rainfall (mm)": "{:.0f}", "Rain vs average (%)": lambda v: "0" if f"{v:+.0f}" in ("+0", "-0") else f"{v:+.0f}",
                       "Max temp (°C)": "{:.1f}", "Max temp vs average (°C)": lambda v: "0.0" if f"{v:+.1f}" in ("+0.0", "-0.0") else f"{v:+.1f}", "Days >30 °C": "{:.0f}"},
              gradient={"Local rank": ("RdYlGn", 1, 5), "Rain vs average (%)": "BrBG", "Max temp vs average (°C)": "coolwarm"})

> 💬 **Questions to discuss — Task 2.2 and 2.3**
>
> 1. Set METRIC to seasonal_rainfall_mm and then mean_tmax_c (in 2B and 2C). Does each correlation get stronger, weaker or stay similar with local ranks? Why might that be?
> 2. Find a low-yield year with near-average rainfall and temperature, and a low-yield year with unusually low rainfall, unusually high temperature, or both.
> 3. What might explain the first example? Think about rainfall timing, differences within the country, non-weather influences and data uncertainty.

### 3. Forecast and observed weather groups with yield

> 🎯 **Objective:** find out whether seasonal forecasts, issued at the start of the season, can help tell good harvests from poor ones.
>
> **You will learn to:**
> - group years into below-normal, near-normal and above-normal rainfall or temperature
> - compare yield across those groups, first using what actually happened, then using the forecast
> - judge how much the groups overlap, and what a fair risk message would say
>
> **Task sheet:** Task 3

**In short:** If dry years usually bring poor harvests, a forecast of a dry season is a warning. This section checks how clear that link is, first with **observed** weather and then with the **forecast** made at the start of the season.

#### A. The forecast data

Forecasts of **seasonal rainfall** and **average daily maximum temperature**, made on 1 November (1 May for Nigeria) for the coming season. Each forecast is a set of slightly different model runs (*ensemble members*); their average is the *ensemble mean*.

<details><summary><b>More detail</b> (click to open)</summary>

The data contain forecasts of **seasonal total rainfall** and **seasonal mean daily maximum temperature** alongside 'observations' from TAMSAT and ERA5.

- **Lead zero:** initialised on 1 November for November–April (southern countries); 1 May for May–September (Nigeria). 
- Rainfall: convert each monthly mean rate from m/s to mm using the actual number of seconds in the target month, then sum. Temperature: convert K to °C and average monthly daily-maximum means, weighted by days in each month.
- Each country CSV has one row per year, ensemble means and standard deviations first, then individual members. Standard deviation describes ensemble spread. 
- Both forecasts and observations represent whole-country averages, with no crop mask. Forecast grid centres inside the country are weighted by cosine of latitude. The forecast grid is coarser than the observation grids; spatial sampling can contribute to differences.

These are retrospective comparisons. Model bias and ensemble spread both matter; a single case does not establish forecast skill. Daily hot-day counts and SPI are not part of this forecast section.

</details>

In [ ]:
df, monthly = load_country(COUNTRY)
forecast, comparison = load_forecasts(COUNTRY, df)
preview = forecast[["harvest_year", "initialization_date", "rainfall_mm_ensemble_mean", "rainfall_mm_ensemble_sd",
                    "mean_tmax_c_ensemble_mean", "mean_tmax_c_ensemble_sd", "ensemble_members_available"]].head()
preview.columns = ["Harvest year", "Forecast made", "Rainfall: ensemble mean (mm)", "Rainfall: spread (SD, mm)",
                   "Max temp: ensemble mean (°C)", "Max temp: spread (SD, °C)", "Members"]
results_table(preview, f"{COUNTRY}: the first five seasonal forecasts",
              formats={c: "{:.1f}" for c in preview.columns[2:6]})
key_result("The forecast data", [
    f"<b>{len(forecast)} seasons</b> of forecasts, harvests {forecast.harvest_year.min()}–{forecast.harvest_year.max()}.",
    f"Each made at the start of the season ({pd.Timestamp(forecast.initialization_date.iloc[-1]):%d %B}) with "
    f"<b>{forecast.ensemble_members_available.min()}–{forecast.ensemble_members_available.max()}</b> ensemble members."])

#### B. Yield in below-, near- and above-normal years

For rainfall and then temperature: **left** = years grouped by what was observed, **right** = years grouped by the forecast. **Top** = raw yield, **bottom** = local rank. Each dot is one harvest year.

<details><summary><b>More detail: how the groups are defined</b> (click to open)</summary>

For each weather variable, the left column groups years by observations and the right column by forecast ensemble means. The upper row shows **raw maize yield**; the lower row shows **local yield rank**. Each dot is one harvest year; boxes show the median and middle 50%, with whiskers extending to the most extreme values within 1.5 interquartile ranges. All years are shown as dots, including outliers.

**How the groups are defined:** calculate anomalies relative to each source's own mean over the reference harvest years (default 1991–2020). Then find the 33⅓ and 66⅔ percentiles of those reference anomalies, separately for observations and forecast ensemble means. Values below the lower cutoff are below normal; values above the upper cutoff are above normal; values on or between the cutoffs are near normal. The cutoff table reports their values. “Near normal” means the middle tercile, not necessarily an anomaly of zero.

The forecast groups use the historical distribution of **ensemble means**, not individual-member terciles or forecast probabilities. Using each source's own cutoffs accommodates differences in its mean and spread. Subtracting a fixed mean does not change which years fall into each source's terciles.

Within each weather comparison, both columns use exactly the same years with complete forecast, observation and yield data. Their assignments to groups can differ. Group sizes need not be equal, especially outside the reference period or with tied values.

This is a **retrospective association exercise**, with reference years allowed to overlap the displayed years. It is not an independently validated yield prediction. Raw yield trends, warming and other influences can affect composites. Local ranks reduce sensitivity to gradual yield trends but use neighbouring years, including future years.

</details>

In [ ]:
# Settings — Section 3B
COMPOSITE_START_YEAR = 1984      # harvest years to compare
COMPOSITE_END_YEAR = 2024
COMPOSITE_BASELINE_START = 1991  # reference years used to define "normal"
COMPOSITE_BASELINE_END = 2020

In [ ]:
df, monthly = load_country(COUNTRY)
forecast, comparison = load_forecasts(COUNTRY, df)
if (COMPOSITE_START_YEAR > COMPOSITE_END_YEAR
        or COMPOSITE_BASELINE_START > COMPOSITE_BASELINE_END):
    raise ValueError("Start years must be no later than end years.")
yield_columns = ["yield_t_ha", "local_yield_rank_5"]
composite_data = comparison.merge(df[["harvest_year"] + yield_columns],
                                  on="harvest_year", validate="one_to_one")
group_names = ["Below normal", "Near normal", "Above normal"]
group_colours = ["#6096b4", "#aaaaaa", "#cf9063"]
cutoff_rows, count_rows, assignment_tables = [], [], []

for prefix, observed_column, metric_label, observed_source in forecast_metrics:
    forecast_column = prefix + "_ensemble_mean"
    required = [observed_column, forecast_column] + yield_columns
    complete = composite_data.replace([np.inf, -np.inf], np.nan).dropna(subset=required).copy()
    reference = complete.loc[complete.harvest_year.between(
        COMPOSITE_BASELINE_START, COMPOSITE_BASELINE_END)]
    selected = complete.loc[complete.harvest_year.between(
        COMPOSITE_START_YEAR, COMPOSITE_END_YEAR)].copy()
    if len(reference) < 10 or len(selected) < 3:
        raise ValueError("Use at least ten complete reference years and three comparison years.")
    fig, axes = plt.subplots(2, 2, figsize=(12, 8), sharey="row")
    assignments = selected[["harvest_year"] + yield_columns].copy()
    assignments["weather_metric"] = prefix
    for column, (kind, value_column, source_label) in enumerate([
        ("observed", observed_column, f"Observed ({observed_source})"),
        ("forecast", forecast_column, "Forecast ensemble mean"),
    ]):
        reference_mean = reference[value_column].mean()
        reference_anomalies = reference[value_column] - reference_mean
        lower, upper = reference_anomalies.quantile([1/3, 2/3]).to_numpy()
        if lower >= upper:
            plt.close(fig)
            raise ValueError("Distinct tercile cutoffs cannot be obtained; choose another reference period.")
        anomaly = selected[value_column] - reference_mean
        group = np.select([anomaly < lower, anomaly > upper],
                          [group_names[0], group_names[2]], default=group_names[1])
        assignments[kind + "_anomaly"] = anomaly.to_numpy()
        assignments[kind + "_group"] = group
        cutoff_rows.append({"Weather metric": metric_label, "Grouping source": source_label,
                            "Reference years": len(reference), "Reference mean": reference_mean,
                            "Lower anomaly cutoff": lower, "Upper anomaly cutoff": upper})
        counts = [int(np.sum(group == name)) for name in group_names]
        count_rows.append({"Weather metric": metric_label, "Grouping source": source_label,
                           **dict(zip(group_names, counts)), "Matched years": len(selected)})
        assert sum(counts) == len(selected)
        for row_number, (yield_column, ylabel) in enumerate([
            ("yield_t_ha", "Maize yield (tonnes/ha)"),
            ("local_yield_rank_5", "Local yield rank (1–5)"),
        ]):
            ax = axes[row_number, column]
            for position, (name, colour) in enumerate(zip(group_names, group_colours), start=1):
                values = selected.loc[group == name, yield_column].to_numpy(dtype=float)
                if len(values):
                    bp = ax.boxplot([values], positions=[position], widths=.5,
                                    patch_artist=True, showfliers=False,
                                    medianprops={"color": "black", "linewidth": 1.5})
                    bp["boxes"][0].set(facecolor=colour, alpha=.35)
                    offsets = np.linspace(-.15, .15, len(values)) if len(values) > 1 else np.zeros(1)
                    ax.scatter(position + offsets, values, s=24, color=colour, alpha=.85,
                               edgecolor="white", linewidth=.3, zorder=3)
            ax.set_xticks([1, 2, 3], [f"{name}\n(n={n})" for name, n in zip(group_names, counts)])
            ax.set_xlim(.5, 3.5); ax.set_ylabel(ylabel); ax.grid(axis="y", alpha=.2)
            if row_number == 0:
                ax.set_title(source_label)
            else:
                ax.set_ylim(.7, 5.3); ax.set_yticks([1, 2, 3, 4, 5])
    fig.suptitle(f"{COUNTRY}: yield by {metric_label.lower()} tercile\n"
                 f"Harvests {COMPOSITE_START_YEAR}–{COMPOSITE_END_YEAR} | "
                 f"Reference {COMPOSITE_BASELINE_START}–{COMPOSITE_BASELINE_END} | "
                 f"{len(selected)} matched years", fontsize=12)
    plt.tight_layout(rect=(0, 0, 1, .94)); plt.show()
    assignment_tables.append(assignments)

composite_cutoffs = pd.DataFrame(cutoff_rows)
composite_counts = pd.DataFrame(count_rows)
composite_assignments = pd.concat(assignment_tables, ignore_index=True)

In [ ]:
group_rows = []
for prefix, observed_column, metric_label, source in forecast_metrics:
    part = composite_assignments[composite_assignments.weather_metric.eq(prefix)]
    for kind, name in [("observed", f"Observed ({source})"), ("forecast", "Forecast")]:
        row = {"Weather": metric_label, "Years grouped by": name}
        for g in group_names:
            in_group = part[kind + "_group"].eq(g)
            row[f"{g}: years"] = int(in_group.sum())
            row[f"{g}: mean rank"] = part.loc[in_group, "local_yield_rank_5"].mean()
        group_rows.append(row)
group_table = pd.DataFrame(group_rows)
rank_columns = [c for c in group_table if c.endswith("mean rank")]
results_table(group_table, f"{COUNTRY}: average local yield rank (1 = locally lowest, 5 = highest) in each weather group",
              formats={c: "{:.1f}" for c in rank_columns},
              gradient={tuple(rank_columns): ("RdYlGn", 1, 5)})

lines = []
for prefix, observed_column, metric_label, source in forecast_metrics:
    t = group_table[group_table.Weather.eq(metric_label)].set_index("Years grouped by")
    gap = (t["Above normal: mean rank"] - t["Below normal: mean rank"])
    obs_gap, fc_gap = gap.iloc[0], gap.iloc[1]
    gap_text = lambda g: f"{g:+.1f}" if np.isfinite(g) else "not available (no years in one group)"
    lines.append(f"<b>{metric_label}</b>: grouped by observations, above-normal years average rank "
                 f"{num(t['Above normal: mean rank'].iloc[0], 1)} vs {num(t['Below normal: mean rank'].iloc[0], 1)} in below-normal years "
                 f"(gap {gap_text(obs_gap)}); grouped by the forecast, the gap is <b>{gap_text(fc_gap)}</b>.")
lines.append("The <b>size</b> of a gap, ignoring its sign, shows how far apart the average ranks are. For temperature the gap is "
             "usually negative: hotter years had lower ranks. These are differences in averages; the boxes show how much the groups overlap.")
key_result("Key result: do the weather groups separate good and poor harvests?", lines)
results_table(composite_cutoffs, "How 'normal' was defined: tercile cut-offs (anomaly from the reference mean)",
              formats={"Reference mean": "{:.1f}", "Lower anomaly cutoff": "{:.1f}", "Upper anomaly cutoff": "{:.1f}"})
# composite_assignments holds each year's anomaly, group and yield.

> 💬 **Questions to discuss — Task 3**
>
> 1. **Observed weather.** Compare yield across the observed rainfall and temperature groups. Are poor harvests more common in dry or hot seasons? How much do the groups overlap?
> 2. **Forecast groups.** Do the same patterns appear when years are grouped by the forecast? Is the separation clearer for rainfall or for temperature? Check the number of years in each group.
> 3. **Raw yield or local rank.** Does your interpretation change when you use local rank? Why might that be?
> 4. **A risk message.** Suppose a forecast says below-normal rainfall or above-normal temperature. What do these results suggest about the risk of a poor harvest? What would be too strong a conclusion? (These composites are not calibrated probabilities of low yield.)

### 4. More weather measures: hot days and SPI

> 🎯 **Objective:** try two measures that may capture crop stress better than seasonal totals and averages.
>
> **You will learn to:**
> - read a country-average count of **hot days** above 30, 32 or 35 °C
> - understand **SPI-3**, a standard drought index, and how it differs from a rainfall total
> - check whether these measures line up with yield more closely than rainfall or temperature
>
> **Task sheet:** Task 4

**In short:** A season can have average rainfall but a long dry spell, or an average temperature but a run of very hot days. Hot-day counts and SPI try to capture that kind of stress.

#### A. Hot days through time

The number of days above each temperature, averaged over the country.

<details><summary><b>More detail</b> (click to open)</summary>

The same seasonal anomaly may give different counts of days > 30°C, 32°C and 35°C.

**Country-average hot-day count**: count threshold exceedances at each grid cell first, then average the counts across the country using cell-area weights. For example, 20 days over half the country and 10 over the other half gives 15 days. Fractional counts are valid. This is not the number of days when the national-average temperature exceeded the threshold. The counts describe area-mean exposure; they do not imply that every field experienced those days, or that all crops were sensitive at the same time.

</details>

In [ ]:
fig, ax = plt.subplots(figsize=(11, 3.5))
for t in [30, 32, 35]:
    ax.plot(df.harvest_year, df[f"hot_days_gt{t}c"], "o-", ms=3, label=f">{t}°C")
ax.set_ylabel("Country-average hot-day count"); ax.set_xlabel("Harvest year"); ax.legend(); ax.grid(alpha=.2)
plt.tight_layout(); plt.show()

In [ ]:
recent = df[["harvest_year", "yield_t_ha", "local_yield_rank_5", "hot_days_gt30c", "hot_days_gt32c", "hot_days_gt35c"]].tail(10)
recent.columns = ["Harvest year", "Yield (t/ha)", "Local rank", "Days >30 °C", "Days >32 °C", "Days >35 °C"]
results_table(recent, f"{COUNTRY}: the last ten seasons (redder = more hot days)",
              formats={"Yield (t/ha)": "{:.2f}", "Local rank": "{:g}", "Days >30 °C": "{:.1f}", "Days >32 °C": "{:.1f}", "Days >35 °C": "{:.1f}"},
              gradient={"Days >30 °C": "Oranges", "Days >32 °C": "Oranges", "Days >35 °C": "Oranges"})
first, last = df.head(10), df.tail(10)
hottest = df.loc[df.hot_days_gt30c.idxmax()]
key_result("Key result: hot days", [
    f"Days above 30 °C averaged <b>{first.hot_days_gt30c.mean():.0f}</b> per season in {first.harvest_year.min()}–{first.harvest_year.max()} "
    f"and <b>{last.hot_days_gt30c.mean():.0f}</b> in {last.harvest_year.min()}–{last.harvest_year.max()}.",
    f"The most days above 30 °C: <b>{hottest.hot_days_gt30c:.0f}</b> in {hottest.harvest_year:.0f} "
    f"(yield {hottest.yield_t_ha:.2f} t/ha, local rank {hottest.local_yield_rank_5:g})."])

#### B. Hot days in one season

How one season compares with the 1991–2020 average at each threshold.

<details><summary><b>More detail</b> (click to open)</summary>

HEAT_EXAMPLE_YEAR selects the harvest year. The first panel shows the actual country-average number of hot days at each threshold alongside its 1991–2020 seasonal average. The second shows the percentage departure from that average: **100 × (selected count − reference count) / reference count**.

These are counts, not the percentage of season days. Percentage anomalies can be large when the reference count is small; a zero reference gives an undefined percentage anomaly, shown as missing.

</details>

In [ ]:
# Settings — Section 4B
HEAT_EXAMPLE_YEAR = 2016  # harvest year, 1984–2024

In [ ]:
if HEAT_EXAMPLE_YEAR not in df.harvest_year.values:
    raise ValueError("Choose a harvest year present in the country CSV.")
thresholds = [30, 32, 35]
heat_fields = [f"hot_days_gt{t}c" for t in thresholds]
heat_row = df.loc[df.harvest_year == HEAT_EXAMPLE_YEAR].iloc[0]
reference = df[df.harvest_year.between(1991, 2020)][heat_fields].mean()
actual = heat_row[heat_fields].astype(float)
percent_anomaly = 100 * (actual - reference) / reference.where(reference > 0)
heat_summary = pd.DataFrame({"Threshold (°C)": thresholds,
    "Selected season: country-average days": actual.to_numpy(),
    "1991–2020 seasonal mean days": reference.to_numpy(),
    "Anomaly (%)": percent_anomaly.to_numpy()})
x = np.arange(len(thresholds))
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
axes[0].bar(x - .18, actual, width=.36, color="#d36c17", label=str(HEAT_EXAMPLE_YEAR))
axes[0].bar(x + .18, reference, width=.36, color="#999999", label="1991–2020 seasonal mean")
axes[0].set_ylabel("Country-average hot-day count (days)"); axes[0].set_title("Absolute hot-day counts"); axes[0].legend(fontsize=9)
axes[1].bar(x, percent_anomaly, color=["#367a9c" if v < 0 else "#d36c17" for v in percent_anomaly])
axes[1].axhline(0, color="black", lw=1)
axes[1].set_ylabel("Anomaly from seasonal mean (%)"); axes[1].set_title("Percentage anomalies")
for i, value in enumerate(percent_anomaly):
    if not np.isfinite(value): axes[1].text(i, 0, "Undefined", ha="center", va="bottom")
for ax in axes:
    ax.set_xticks(x, [f">{t}°C" for t in thresholds]); ax.set_xlabel("Daily maximum temperature threshold")
    ax.grid(axis="y", alpha=.2); ax.set_axisbelow(True)
fig.suptitle(f"{COUNTRY} | Harvest {HEAT_EXAMPLE_YEAR} | {heat_row.season_start} to {heat_row.season_end}")
plt.tight_layout(); plt.show()
heat_reference, heat_actual, heat_anomaly = reference, actual, percent_anomaly

In [ ]:
results_table(heat_summary, f"{COUNTRY}, harvest {HEAT_EXAMPLE_YEAR}: hot days compared with 1991–2020",
              formats={c: "{:.1f}" for c in heat_summary.columns[1:]} | {"Threshold (°C)": "{:.0f}"},
              gradient={"Anomaly (%)": "coolwarm"})
key_result(f"Key result: hot days in {HEAT_EXAMPLE_YEAR}", [
    f"Above {t} °C: <b>{a:.0f} days</b> against a usual {r:.0f} ("
    + (f"{p:+.0f} %" if np.isfinite(p) else "no usual count") + ")."
    for t, a, r, p in zip(thresholds, heat_actual, heat_reference, heat_anomaly)] +
    [f"Yield that season: {heat_row.yield_t_ha:.2f} t/ha, local rank {heat_row.local_yield_rank_5:g} of 5."])

#### C. Understanding SPI

**SPI** (Standardised Precipitation Index) shows how unusually wet or dry it was compared with normal for that place and time of year: **negative = drier than normal, positive = wetter**. SPI-3 uses three months of rain.

<details><summary><b>More detail</b> (click to open)</summary>

SPI standardises rainfall to highlight unusually wet or dry conditions relative to the local climate and time of year.

SPI-3 describes rainfall over a moving three-month window. We average the SPI-3 values ending within the growing season, so the early values also capture antecedent rainfall before the season began.

We therefore expect mean seasonal SPI-3 and seasonal rainfall anomalies to show similar, but not identical, patterns. SPI-3 includes antecedent rainfall, uses overlapping windows and standardises rainfall separately for each time of year. Negative values indicate predominantly relatively dry conditions across those windows; positive values indicate relatively wet conditions. 

The plots below compare **seasonal rainfall anomalies (mm relative to the 1991–2020 mean)** with **mean seasonal SPI-3**, indexed by harvest year.

Background: [WMO SPI User Guide](https://www.droughtmanagement.info/literature/WMO_standardized_precipitation_index_user_guide_en_2012.pdf).

</details>

In [ ]:
# Settings — Section 4C
SPI_START_YEAR = 1984
SPI_END_YEAR = 2024

In [ ]:
if SPI_START_YEAR > SPI_END_YEAR:
    raise ValueError("The start year must be no later than the end year.")
spi_series = df.loc[df.harvest_year.between(SPI_START_YEAR, SPI_END_YEAR),
                   ["harvest_year", "seasonal_rainfall_mm", "mean_spi3"]].sort_values("harvest_year")
if spi_series.empty:
    raise ValueError("Choose harvest years covered by the data (1984–2024).")
years = spi_series.harvest_year.to_numpy()
season_label = "May–September" if COUNTRY == "Nigeria" else "November–April"
fig, axes = plt.subplots(2, 1, figsize=(12, 7), sharex=True)
rainfall_reference = df.loc[df.harvest_year.between(1991, 2020), "seasonal_rainfall_mm"].mean()
rainfall_anomaly = spi_series.seasonal_rainfall_mm - rainfall_reference
axes[0].bar(years, rainfall_anomaly,
            color=["#b36b38" if v < 0 else "#367a9c" for v in rainfall_anomaly])
axes[0].axhline(0, color="black", lw=1)
axes[0].set_ylabel("Seasonal rainfall anomaly (mm)")
axes[0].set_title(f"{season_label} rainfall minus the 1991–2020 mean ({rainfall_reference:.1f} mm)")
axes[1].bar(years, spi_series.mean_spi3,
            color=["#b36b38" if v < 0 else "#367a9c" for v in spi_series.mean_spi3])
axes[1].axhline(0, color="black", lw=1)
axes[1].set_ylabel("Mean seasonal SPI-3 (dimensionless)")
axes[1].set_title(f"Mean of monthly SPI-3 values ending during {season_label}")
axes[1].set_xlabel("Harvest year")
for ax in axes:
    ax.grid(axis="y", alpha=.2); ax.set_axisbelow(True)
fig.suptitle(f"{COUNTRY}: seasonal rainfall anomalies and mean SPI-3 | Reference period 1991–2020")
plt.tight_layout(); plt.show()
if spi_series[["seasonal_rainfall_mm", "mean_spi3"]].isna().any().any():
    print("Missing values appear as gaps.")

In [ ]:
both = pd.DataFrame({"year": years, "rain": rainfall_anomaly.to_numpy(), "spi": spi_series.mean_spi3.to_numpy()}).dropna()
r_spi = both.rain.corr(both.spi)
differ = both.loc[np.sign(both.rain) != np.sign(both.spi), "year"].astype(int).tolist()
key_result("Key result: rainfall and SPI", [
    (f"Seasonal rainfall anomaly and mean SPI-3 over {len(both)} seasons: <b>r = {r_spi:.2f}</b> ({strength(r_spi)})."
     if len(both) >= 3 and np.isfinite(r_spi) else "Too few seasons in this range to compare rainfall and SPI-3."),
    f"In <b>{len(differ)} of {len(both)}</b> seasons they disagree on wet or dry"
    + (f": {', '.join(map(str, differ))}." if differ else ".") +
    " SPI-3 also counts rain just before the season and treats each time of year separately, so they need not agree."])

> 💬 **Questions to discuss — Task 4.1**
>
> 1. What does SPI-3 measure, and how is mean seasonal SPI-3 calculated?
> 2. Why should mean seasonal SPI-3 and seasonal rainfall anomalies be similar but not identical?
> 3. What does a country-average count of days above a temperature threshold represent?

#### D. All four measures against yield

Four measures side by side, against local rank (scatter) and by yield group (boxes). The single-measure tools in Section 2 also accept SPI-3 and hot-day counts.

In [ ]:
# Settings — Section 4D: choose exactly four measures
METRICS = ["seasonal_rainfall_mm", "mean_spi3", "mean_tmax_c", "hot_days_gt30c"]
# options: seasonal_rainfall_mm, mean_spi3, mean_tmax_c, hot_days_gt30c, hot_days_gt32c, hot_days_gt35c

In [ ]:
if len(METRICS) != 4 or any(m not in labels for m in METRICS):
    raise ValueError("Choose four metrics from labels (shown in setup).")
fig, axes = plt.subplots(2, 2, figsize=(11, 7))
for ax, metric in zip(axes.flat, METRICS):
    pairs = df[[metric, "local_yield_rank_5"]].dropna()
    ax.scatter(pairs[metric], pairs.local_yield_rank_5, s=30, alpha=.75)
    ax.set_xlabel(labels[metric]); ax.set_ylabel("Local yield rank"); ax.set_yticks(range(1, 6)); ax.grid(alpha=.2)
    ax.set_title(f"r = {pairs[metric].corr(pairs.local_yield_rank_5):.2f}; {len(pairs)} years")
plt.tight_layout(); plt.show()

In [ ]:
category_order, rank_order, category_definitions = yield_groups(df)
fig, axes = plt.subplots(4, 2, figsize=(12, 14), sharey="row")
for i, metric in enumerate(METRICS):
    weather_boxes(axes[i, 0], df, metric, "local_yield_rank_5", rank_order)
    weather_boxes(axes[i, 1], df, metric, "absolute_yield_category", category_order)
axes[0, 0].set_title("Grouped by local yield rank")
axes[0, 1].set_title("Grouped by absolute yield")
fig.suptitle(f"{COUNTRY}: weather distributions by yield group", fontsize=15)
plt.tight_layout(rect=(0, 0, 1, .97)); plt.show()

In [ ]:
r_table = pd.DataFrame([{"Weather measure": labels[m], "Years": len(df[[m, "local_yield_rank_5"]].dropna()),
                         "r with local rank": df[[m, "local_yield_rank_5"]].dropna().corr().iloc[0, 1]} for m in METRICS])
r_table["Strength"] = r_table["r with local rank"].map(strength)
r_table = r_table.reindex(r_table["r with local rank"].abs().sort_values(ascending=False).index)
results_table(r_table, f"{COUNTRY}: which measure lines up best with local yield rank? (strongest first)",
              formats={"r with local rank": "{:+.2f}"}, gradient={"r with local rank": ("PuOr", -1, 1)})
best = r_table.iloc[0]
key_result("Key result: the measure that lines up best", [
    f"<b>{best['Weather measure']}</b>: r = {best['r with local rank']:+.2f} ({best.Strength}).",
    "Negative r means higher values go with lower yields (expected for heat); positive r means higher values go with higher yields (expected for rain in dry places)."])

> 💬 **Questions to discuss — Task 4.2 and 4.3**
>
> 1. Do SPI-3 and hot-day counts show relationships with local rank that are less clear with seasonal rainfall or average temperature?
> 2. Compare the counts above 30, 32 and 35 °C. Set HEAT_EXAMPLE_YEAR in 4B, and swap the hot-day measure in the 4D METRICS list. Does the relationship with yield depend on the threshold?

### 5. Within the season, month by month

> 🎯 **Objective:** look inside one season to see *when* it was dry or hot.
>
> **You will learn to:**
> - compare monthly rainfall, temperature, SPI-3 and hot days with the 1991–2020 normal
> - spot a season with near-average totals but a dry or hot spell at a bad time
> - think about why the timing of stress matters for crops
>
> **Task sheet:** Task 5

**In short:** Seasonal totals can hide a lot. A dry month at flowering can cut yields even when the season's total rain looks normal.

<details><summary><b>More detail</b> (click to open)</summary>

**DETAIL_YEAR** selects the harvest year and **HEAT_THRESHOLD_C** selects 30, 32 or 35°C (the thresholds supplied in the CSVs). The panels show monthly rainfall, SPI-3, daily maximum temperature, heat exposure and rainfall/temperature anomalies. The yield summary shows **actual yield, anomaly from the country's 1984–2024 mean, and local yield rank**. The anomaly is shown in tonnes/ha and percent; it is not detrended and differs from the local rank.

Dashed grey lines on the rainfall, temperature and heat panels show the 1991–2020 average for each calendar month. SPI-3 has a zero reference and the selected season's mean. Rainfall anomalies (mm) and temperature anomalies (°C) are differences from each calendar month's 1991–2020 average; they are not percentage or standardised anomalies.

Each monthly SPI-3 includes its ending month and the preceding two months. November SPI-3 therefore includes September–November rainfall. The monthly SPI windows overlap.

Monthly normals here use calendar years 1991–2020, so a season's total anomaly can differ slightly from Section 4C, which uses harvest years 1991–2020.

</details>

In [ ]:
# Settings — Section 5
DETAIL_YEAR = 2016       # harvest year, 1984–2024
HEAT_THRESHOLD_C = 30    # 30, 32 or 35 °C

In [ ]:
if HEAT_THRESHOLD_C not in (30, 32, 35):
    raise ValueError("Choose 30, 32 or 35°C: other thresholds are not in the supplied CSV.")
if DETAIL_YEAR not in df.harvest_year.values:
    raise ValueError(f"Choose a harvest year from {df.harvest_year.min()} to {df.harvest_year.max()}.")
heat_column = f"hot_days_gt{HEAT_THRESHOLD_C:g}c"
row = df.loc[df.harvest_year == DETAIL_YEAR].iloc[0]
mean_yield = df.yield_t_ha.mean()
anomaly = row.yield_t_ha - mean_yield
anomaly_pct = 100 * anomaly / mean_yield
summary = pd.DataFrame({"Measure": ["Yield (tonnes/ha)", "1984–2024 mean yield (tonnes/ha)", "Yield anomaly (tonnes/ha)", "Yield anomaly (%)", "Local yield rank (1–5)"],
                        "Value": [row.yield_t_ha, mean_yield, anomaly, anomaly_pct, row.local_yield_rank_5]})
start, end = pd.Timestamp(row.season_start), pd.Timestamp(row.season_end)
selected = monthly[monthly.month.between(start, end)].copy()
assert np.isclose(selected.spi3.mean(), row.mean_spi3)
baseline = monthly[monthly.month.dt.year.between(1991, 2020)]
climatology = baseline.groupby(baseline.month.dt.month)[["rainfall_mm", "mean_tmax_c", heat_column]].mean()
ref = climatology.reindex(selected.month.dt.month)
selected["rainfall_anomaly_mm"] = selected.rainfall_mm.to_numpy() - ref.rainfall_mm.to_numpy()
selected["tmax_anomaly_c"] = selected.mean_tmax_c.to_numpy() - ref.mean_tmax_c.to_numpy()
columns = ["month", "rainfall_mm", "rainfall_anomaly_mm", "spi3", "mean_tmax_c", "tmax_anomaly_c", heat_column]
table = selected[columns].copy()
numeric_columns = table.select_dtypes(include="number").columns
table[numeric_columns] = table[numeric_columns].round(2)
x = np.arange(len(selected))
month_labels = selected.month.dt.strftime("%b %Y")
fig, axes = plt.subplots(3, 2, figsize=(12, 11))
ax = axes[0, 0]
ax.bar(x, selected.rainfall_mm, color="#367a9c", label="Selected season")
ax.plot(x, ref.rainfall_mm.to_numpy(), "o--", color="#555555", ms=4, label="1991–2020 monthly mean")
ax.set_ylabel("Rainfall (mm/month)"); ax.set_title("Monthly rainfall"); ax.legend(fontsize=8)
ax = axes[2, 0]
ax.bar(x, selected.spi3, color=["#b36b38" if v < 0 else "#367a9c" for v in selected.spi3])
ax.axhline(0, color="black", lw=1)
ax.axhline(row.mean_spi3, color="#555555", ls="--", label=f"Season mean = {row.mean_spi3:.2f}")
ax.set_ylabel("SPI-3"); ax.set_title("Monthly SPI-3"); ax.legend(fontsize=8)
ax = axes[1, 0]
ax.plot(x, selected.mean_tmax_c, "o-", color="#d36c17", label="Selected season")
ax.plot(x, ref.mean_tmax_c.to_numpy(), "o--", color="#555555", ms=4, label="1991–2020 monthly mean")
ax.set_ylabel("Mean daily maximum temperature (°C)"); ax.set_title("Monthly temperature"); ax.legend(fontsize=8)
ax = axes[2, 1]
ax.bar(x, selected[heat_column], color="#d36c17", label="Selected season")
ax.plot(x, ref[heat_column].to_numpy(), "o--", color="#555555", ms=4, label="1991–2020 monthly mean")
ax.set_ylabel(f"Country-average days >{HEAT_THRESHOLD_C:g}°C"); ax.set_title("Monthly heat exposure"); ax.legend(fontsize=8)
ax = axes[0, 1]
ax.bar(x, selected.rainfall_anomaly_mm, color=["#b36b38" if v < 0 else "#367a9c" for v in selected.rainfall_anomaly_mm])
ax.axhline(0, color="black", lw=1)
ax.set_ylabel("Rainfall anomaly (mm)"); ax.set_title("Rainfall anomaly from 1991–2020")
ax = axes[1, 1]
ax.bar(x, selected.tmax_anomaly_c, color=["#367a9c" if v < 0 else "#d36c17" for v in selected.tmax_anomaly_c])
ax.axhline(0, color="black", lw=1)
ax.set_ylabel("Temperature anomaly (°C)"); ax.set_title("Temperature anomaly from 1991–2020")
for ax in axes.flat:
    ax.set_xticks(x, month_labels, rotation=25, ha="right")
    ax.grid(axis="y", alpha=.2); ax.set_axisbelow(True)
fig.suptitle(f"{COUNTRY} | Harvest {DETAIL_YEAR}" + chr(10) +
             f"Yield {row.yield_t_ha:.2f} t/ha | Anomaly {anomaly:+.2f} t/ha ({anomaly_pct:+.1f}%) | Local rank {row.local_yield_rank_5:g}/5", fontsize=13)
plt.tight_layout(rect=(0, 0, 1, .93)); plt.show()
season_detail = {'summary': summary, 'table': table, 'selected': selected, 'row': row, 'anomaly_pct': anomaly_pct}

In [ ]:
selected, row, anomaly_pct = season_detail["selected"], season_detail["row"], season_detail["anomaly_pct"]
results_table(season_detail["summary"], f"{COUNTRY}, harvest {row.harvest_year:.0f}: yield", formats={"Value": "{:.2f}"})
table_shown = season_detail["table"].copy()
table_shown["month"] = table_shown.month.dt.strftime("%b %Y")
table_shown.columns = ["Month", "Rainfall (mm)", "Rainfall vs normal (mm)", "SPI-3", "Max temp (°C)",
                       "Max temp vs normal (°C)", f"Days >{HEAT_THRESHOLD_C:g} °C"]
results_table(table_shown, f"{COUNTRY}, harvest {row.harvest_year:.0f}: month by month (brown = drier, blue = wetter; red = hotter)",
              formats={c: "{:.1f}" for c in table_shown.columns[1:]} | {"SPI-3": "{:+.2f}"},
              gradient={"Rainfall vs normal (mm)": "BrBG", "SPI-3": "BrBG", "Max temp vs normal (°C)": "coolwarm"})
season_rain = selected.rainfall_anomaly_mm.sum()
dry = selected.loc[selected.rainfall_anomaly_mm.idxmin()]
hot = selected.loc[selected.tmax_anomaly_c.idxmax()]
key_result(f"Key result: {row.harvest_year:.0f} month by month", [
    f"Season rainfall was <b>{abs(season_rain):.0f} mm {'below' if season_rain < 0 else 'above'}</b> the 1991–2020 normal "
    f"({int((selected.rainfall_anomaly_mm < 0).sum())} of {len(selected)} months drier than normal).",
    f"Driest month compared with normal: <b>{dry.month:%B %Y}</b> ({dry.rainfall_anomaly_mm:+.0f} mm).",
    f"Hottest month compared with normal: <b>{hot.month:%B %Y}</b> ({hot.tmax_anomaly_c:+.1f} °C).",
    f"Yield: {row.yield_t_ha:.2f} t/ha ({anomaly_pct:+.0f} % against the long-term average), local rank {row.local_yield_rank_5:g} of 5."])

> 💬 **Questions to discuss — Task 5**
>
> 1. Pick a low-ranked and a high-ranked year from Task 2 and run this section for each. Compare their monthly rainfall, temperature, SPI-3 and hot days.
> 2. Were dry or hot conditions concentrated in particular months, or did they last all season? Can you find a season with near-average total rainfall but a dry spell?
> 3. How might the timing of these conditions affect crops? What information about planting dates and crop development would help you interpret the results?

### 6. El Niño and La Niña

> 🎯 **Objective:** see how El Niño and La Niña years relate to weather and yield in your country.
>
> **You will learn to:**
> - group years into El Niño, neutral and La Niña using the ONI index
> - compare yield, rainfall, temperature and hot days across the groups
> - see how a stricter definition of El Niño changes the result and the number of years
>
> **Task sheet:** Task 6

**In short:** El Niño and La Niña are changes in Pacific Ocean temperatures that shift rainfall around the world. In southern Africa, El Niño often brings drier, hotter seasons. Here you can check how often that showed up in yield.

<details><summary><b>More detail</b> (click to open)</summary>

The settings control **El Niño, neutral and La Niña** groups shown in a table and box plots for yield, local yield rank, temperature, rainfall and hot-day count.

**What is the anomaly?** The provided time series of ONI is the three-month mean sea-surface temperature anomaly in the Niño 3.4 region (5°N–5°S, 170°W–120°W). Its units are °C relative to NOAA's season-specific, centred 30-year reference periods, updated every five years. It is not a local air-temperature anomaly, and its reference is not the fixed 1991–2020 weather baseline used elsewhere here.

With the default threshold of **0.5°C**, the selected three-month value defines El Niño at ONI ≥ +0.5°C, La Niña at ONI ≤ −0.5°C, and neutral at −0.5°C < ONI < +0.5°C. This practical groups single-season index values; the traditional historical ONI episode definition also requires at least five consecutive overlapping seasons beyond the threshold. We do not impose that persistence rule here.

If you raise the ENSO threshold, the neutral band stays fixed at ±0.5°C: weaker positive/negative years are **excluded**, not relabelled neutral. Sample sizes change and are shown explicitly.

Southern Africa defaults to OND of the year before harvest. Nigeria defaults to MAM of the harvest year as an exploratory pre-season context. Changing the index season does not change the agricultural growing season (May - September for Nigeria; November - April for southern African countries).

</details>

In [ ]:
# Settings — Section 6
INDEX_SEASON = "MAM" if COUNTRY == "Nigeria" else "OND"   # three-month ONI season
INDEX_YEAR_OFFSET = 0 if COUNTRY == "Nigeria" else -1     # relative to harvest year
ENSO_THRESHOLD_C = 0.5      # try 0.5, 1.0, 1.5 or 2.0; neutral stays within ±0.5 °C
ENSO_HEAT_THRESHOLD_C = 30  # 30, 32 or 35 °C

In [ ]:
oni = pd.read_csv(DATA_DIR / "oni_three_month.csv")
phase_order = ["El Niño", "Neutral", "La Niña"]
phase_colors = ["#d98d64", "#bdbdbd", "#79afce"]

def explore_enso(index_season, year_offset, threshold_c, heat_threshold_c):
    if index_season not in oni.SEAS.unique():
        raise ValueError("Choose a three-month season present in the ONI file.")
    if not np.isfinite(threshold_c) or threshold_c < 0.5:
        raise ValueError("Choose an ENSO threshold >=0.5°C; neutral is fixed within ±0.5°C.")
    if heat_threshold_c not in (30, 32, 35):
        raise ValueError("Heat threshold must be 30, 32 or 35°C.")
    if int(year_offset) != year_offset:
        raise ValueError("The index-year offset must be an integer.")
    heat_field = f"hot_days_gt{heat_threshold_c:g}c"
    context = df.copy()
    context["index_year"] = context.harvest_year + int(year_offset)
    context = context.merge(oni[oni.SEAS == index_season][["YR", "ANOM"]], left_on="index_year", right_on="YR", how="left", validate="many_to_one")
    context["enso_group"] = np.select(
        [context.ANOM >= threshold_c, context.ANOM <= -threshold_c, context.ANOM.abs() < 0.5, context.ANOM.notna()],
        ["El Niño", "La Niña", "Neutral", "Excluded: weaker index"], default="Missing ONI")
    summary = context.groupby("enso_group").agg(
        years=("harvest_year", "count"), mean_yield_t_ha=("yield_t_ha", "mean"),
        mean_yield_rank=("local_yield_rank_5", "mean"), mean_tmax_c=("mean_tmax_c", "mean"),
        mean_rainfall_mm=("seasonal_rainfall_mm", "mean"), mean_hot_days=(heat_field, "mean"))
    plot_metrics = [("yield_t_ha", "Yield (tonnes/ha)"), ("local_yield_rank_5", "Local yield rank"),
                    ("mean_tmax_c", "Mean daily maximum temperature (°C)"),
                    ("seasonal_rainfall_mm", "Seasonal rainfall (mm)"),
                    (heat_field, f"Country-average days >{heat_threshold_c:g}°C")]
    fig, axes = plt.subplots(3, 2, figsize=(12, 11))
    for ax, (metric, label) in zip(axes.flat, plot_metrics):
        groups = [context.loc[context.enso_group.eq(phase), metric].dropna().to_numpy() for phase in phase_order]
        for position, (values, color) in enumerate(zip(groups, phase_colors), start=1):
            if len(values):
                box = ax.boxplot([values], positions=[position], widths=.55, patch_artist=True,
                    medianprops={"color": "#222222", "linewidth": 1.5},
                    flierprops={"marker": "o", "markersize": 4})
                box["boxes"][0].set_facecolor(color)
            else:
                ax.text(position, .5, "No years", transform=ax.get_xaxis_transform(), ha="center")
        ax.set_xticks([1, 2, 3], [phase + chr(10) + f"(n={len(values)})" for phase, values in zip(phase_order, groups)])
        ax.set_xlim(.5, 3.5); ax.set_ylabel(label); ax.set_title(label)
        ax.grid(axis="y", alpha=.2); ax.set_axisbelow(True)
        if metric == "local_yield_rank_5": ax.set_yticks(range(1, 6)); ax.set_ylim(.5, 5.5)
    ax = axes[2, 1]
    color_map = dict(zip(phase_order, phase_colors))
    ax.bar(context.harvest_year, context.ANOM, color=context.enso_group.map(color_map).fillna("#eeeeee"))
    ax.axhline(threshold_c, color="#b0572c", ls="--", label=f"ENSO threshold ±{threshold_c:g}°C")
    ax.axhline(-threshold_c, color="#367a9c", ls="--")
    ax.axhspan(-.5, .5, color="#bbbbbb", alpha=.12, label="Neutral band")
    ax.axhline(0, color="black", lw=.7); ax.set_xlabel("Harvest year"); ax.set_ylabel("ONI anomaly (°C)")
    ax.set_title(f"{index_season} ONI aligned to harvest years"); ax.legend(fontsize=8)
    fig.suptitle(f"{COUNTRY}: ENSO groups | {index_season}, year offset {int(year_offset):+d} | threshold ±{threshold_c:g}°C", fontsize=13)
    plt.tight_layout(rect=(0, 0, 1, .96)); plt.show()
    return context, summary

context, summary = explore_enso(INDEX_SEASON, INDEX_YEAR_OFFSET, ENSO_THRESHOLD_C, ENSO_HEAT_THRESHOLD_C)
enso_summary, enso_context = summary, context

In [ ]:
summary, context = enso_summary, enso_context
order = phase_order + [g for g in summary.index if g not in phase_order]
shown = summary.reindex(order)
shown["years"] = shown["years"].fillna(0)
shown = shown.reset_index()
shown.columns = ["Group", "Years", "Mean yield (t/ha)", "Mean local rank", "Mean max temp (°C)", "Mean rainfall (mm)",
                 f"Mean days >{ENSO_HEAT_THRESHOLD_C:g} °C"]
shown["Harvest years"] = [", ".join(str(y) for y in context.loc[context.enso_group.eq(g), "harvest_year"]) for g in shown.Group]
results_table(shown, f"{COUNTRY}: {INDEX_SEASON} ONI, threshold ±{ENSO_THRESHOLD_C:g} °C",
              formats={"Years": "{:.0f}", "Mean yield (t/ha)": "{:.2f}", "Mean local rank": "{:.1f}", "Mean max temp (°C)": "{:.1f}",
                       "Mean rainfall (mm)": "{:.0f}", f"Mean days >{ENSO_HEAT_THRESHOLD_C:g} °C": "{:.0f}"},
              colours={"Group": dict(zip(phase_order, phase_colors))})
get = lambda g, c: summary[c].get(g, np.nan)
key_result("Key result: El Niño, neutral and La Niña years", [
    f"El Niño years ({int(np.nan_to_num(get('El Niño', 'years')))}): mean local rank <b>{num(get('El Niño', 'mean_yield_rank'), 1)}</b>, "
    f"rainfall {num(get('El Niño', 'mean_rainfall_mm'), 0)} mm.",
    f"Neutral years ({int(np.nan_to_num(get('Neutral', 'years')))}): mean local rank <b>{num(get('Neutral', 'mean_yield_rank'), 1)}</b>, "
    f"rainfall {num(get('Neutral', 'mean_rainfall_mm'), 0)} mm.",
    f"La Niña years ({int(np.nan_to_num(get('La Niña', 'years')))}): mean local rank <b>{num(get('La Niña', 'mean_yield_rank'), 1)}</b>, "
    f"rainfall {num(get('La Niña', 'mean_rainfall_mm'), 0)} mm.",
    "With so few years in each group, one unusual season can move the averages a lot. Check how much the boxes overlap."])

> 💬 **Questions to discuss — Task 6**
>
> 1. Compare rainfall, temperature, hot days, yield and local rank in El Niño, neutral and La Niña years. Which differences are clearest? How much do the groups overlap?
> 2. Raise ENSO_THRESHOLD_C and rerun. Do the relationships get clearer? How many years are left in each group? (The neutral band stays fixed, so weaker events drop out.)
> 3. Does an El Niño or La Niña mean a poor harvest is certain? What limits what we can say: sample size, timing of the index, other influences on yield?

### 7. How good are the seasonal forecasts?

> 🎯 **Objective:** check how well the start-of-season forecasts matched what actually happened.
>
> **You will learn to:**
> - compare forecasts and observations year by year
> - look at one season in detail: what the forecast said, and where the real outcome fell
> - tell the difference between **correlation** (do they rise and fall together?) and **bias** (is the forecast too wet or too hot on average?)
>
> **Task sheet:** Task 7

**In short:** A forecast is useful for advice only if it has some skill. This section checks the forecasts used in Section 3 against TAMSAT rainfall and ERA5 temperature.

#### A. Forecasts and observations through time

Blue = forecast ensemble mean (shading: the range of 80 % of members); orange = observed.

<details><summary><b>More detail</b> (click to open)</summary>

The lines show ensemble means and observations; shading shows the ensemble 10–90% range. Ensemble spread is not a calibrated confidence interval for the observed outcome.

[ECMWF lead-month convention](https://confluence.ecmwf.int/pages/viewpage.action?pageId=129107123): forecast month 1 is the initialisation month for a first-of-month start.

</details>

In [ ]:
forecast, comparison = load_forecasts(COUNTRY, df)
fig, axes = plt.subplots(2, 1, figsize=(12, 8), sharex=True)
for ax, (prefix, observed_column, label, source) in zip(axes, forecast_metrics):
    member_columns = [c for c in forecast if c.startswith(prefix + "_member_")]
    members = comparison[member_columns]
    low, high = members.quantile(.1, axis=1), members.quantile(.9, axis=1)
    ax.fill_between(comparison.harvest_year, low, high, alpha=.2, color="#367a9c",
                    label="Forecast ensemble 10–90% range")
    ax.plot(comparison.harvest_year, comparison[prefix + "_ensemble_mean"], "o-",
            ms=3, color="#367a9c", label="Forecast ensemble mean")
    ax.plot(comparison.harvest_year, comparison[observed_column], "o-", ms=3,
            color="#b0572c", label=source)
    ax.set_ylabel(label); ax.grid(alpha=.2); ax.legend(fontsize=9)
axes[-1].set_xlabel("Harvest year")
fig.suptitle(f"{COUNTRY}: lead-zero forecasts and observed weather (raw values)")
plt.tight_layout(); plt.show()
print("Available members:", forecast.ensemble_members_available.min(), "to",
      forecast.ensemble_members_available.max())

In [ ]:
lines = []
for prefix, observed_column, label, source in forecast_metrics:
    bias = (comparison[prefix + "_ensemble_mean"] - comparison[observed_column]).mean()
    unit = "mm" if prefix.startswith("rain") else "°C"
    word = ("wetter" if bias > 0 else "drier") if unit == "mm" else ("warmer" if bias > 0 else "cooler")
    lines.append(f"<b>{label}</b>: on average the forecast was <b>{abs(bias):.1f} {unit} {word}</b> than {source}.")
key_result("Key result: forecasts through time", lines + [
    "An average difference like this is a <b>bias</b>. It can be removed by comparing anomalies (Section 7B); it does not by itself mean the forecast has no skill."])

#### B. One season in detail

Each blue dot is one ensemble member; the diamond is their average; the orange **X** is what happened. All values are differences from each source's own 1991–2020 average, so the bias is removed.

<details><summary><b>More detail</b> (click to open)</summary>

FORECAST_YEAR selects the harvest year. To avoid mistaking a model's average bias for a seasonal signal, subtract **each source's own reference mean**: the model mean for forecasts and the TAMSAT/ERA5 mean for observations. Each reference harvest year receives equal weight, even when ensemble sizes differ. The selected year is excluded from both reference means if it falls within the baseline.

Rainfall anomalies are in **mm** and temperature anomalies in **°C**. This removes the reference mean difference; it does not correct variance or calibrate probabilities. The fixed reference period can include years after the example year, so this is a retrospective exercise, not a reconstruction of information available at issue time.

</details>

In [ ]:
# Settings — Section 7B
FORECAST_YEAR = 2016             # try a poor-harvest year you found earlier
FORECAST_BASELINE_START = 1991
FORECAST_BASELINE_END = 2020

In [ ]:
forecast, comparison = load_forecasts(COUNTRY, df)
if FORECAST_YEAR not in forecast.harvest_year.values:
    raise ValueError("Choose a harvest year from 1984 to 2024.")
case = comparison.loc[comparison.harvest_year.eq(FORECAST_YEAR)].iloc[0]
reference = comparison.loc[
    comparison.harvest_year.between(FORECAST_BASELINE_START, FORECAST_BASELINE_END)
    & comparison.harvest_year.ne(FORECAST_YEAR)]
if len(reference) < 10:
    raise ValueError("Choose a reference period containing at least ten other harvest years.")
fig, axes = plt.subplots(1, 2, figsize=(11, 5))
rows = []
for ax, (prefix, observed_column, label, source) in zip(axes, forecast_metrics):
    columns = [c for c in forecast if c.startswith(prefix + "_member_")]
    model_reference = reference[prefix + "_ensemble_mean"].mean()
    observed_reference = reference[observed_column].mean()
    members = case[columns].dropna().to_numpy(dtype=float)
    anomalies = members - model_reference
    observed_anomaly = case[observed_column] - observed_reference
    ax.boxplot([anomalies], positions=[1], widths=.35, showfliers=False)
    ax.scatter(np.linspace(.88, 1.12, len(anomalies)), anomalies, s=20, alpha=.6,
               color="#367a9c", label=f"Forecast members (n={len(anomalies)})")
    ax.scatter([1], [anomalies.mean()], marker="D", color="#367a9c", s=55,
               label="Ensemble mean", zorder=4)
    ax.scatter([1], [observed_anomaly], marker="X", color="#b0572c", s=110,
               label=f"Observed ({source})", zorder=5)
    ax.axhline(0, color="grey", ls="--")
    ax.set_xticks([1], [str(FORECAST_YEAR)])
    ax.set_ylabel(label + " — anomaly")
    ax.grid(axis="y", alpha=.2); ax.legend(fontsize=8)
    rows.append({"Metric": label, "Forecast mean": members.mean(),
                 "Forecast SD": members.std(ddof=1), "Observed": case[observed_column],
                 "Model reference mean": model_reference, "Observed reference mean": observed_reference,
                 "Forecast mean anomaly": anomalies.mean(), "Observed anomaly": observed_anomaly})
fig.suptitle(f"{COUNTRY} | Harvest {FORECAST_YEAR} | Initialised {case.initialization_date}")
plt.tight_layout(); plt.show()
case_rows, case_season, case_reference = rows, case, reference

In [ ]:
rows, case, reference = case_rows, case_season, case_reference
case_table = pd.DataFrame(rows)
results_table(case_table, f"{COUNTRY}, harvest {case.harvest_year:.0f}: forecast and observed "
              f"(reference {reference.harvest_year.min()}–{reference.harvest_year.max()}, {len(reference)} years, excluding {case.harvest_year:.0f})",
              formats={c: "{:.1f}" for c in case_table.columns[1:]})
lines = []
for (prefix, observed_column, label, source), r in zip(forecast_metrics, rows):
    members = case[[c for c in forecast if c.startswith(prefix + "_member_")]].dropna().to_numpy(dtype=float) - r["Model reference mean"]
    share_below = 100 * np.mean(members < r["Observed anomaly"])
    same = np.sign(r["Forecast mean anomaly"]) == np.sign(r["Observed anomaly"])
    lines.append(f"<b>{label}</b>: forecast anomaly {r['Forecast mean anomaly']:+.1f}, observed {r['Observed anomaly']:+.1f} "
                 f"({'same direction' if same else 'opposite direction'}). The observed value was above {share_below:.0f} % of the members.")
key_result(f"Key result: the {case.harvest_year:.0f} season", lines)

> 💬 **Questions to discuss — Task 7.2**
>
> 1. Choose a poor-harvest year found earlier. Did the forecast suggest unusually dry or hot conditions?
> 2. How large was the spread between members, and where did the observed outcome fall?

#### C. Forecast skill over all years

Each dot is one harvest year. Dots near the dashed line mean the forecast matched the observed value.

<details><summary><b>More detail</b> (click to open)</summary>

Each dot represents one harvest year. The plots show the **forecast ensemble mean** against observed seasonal rainfall (TAMSAT) and mean daily maximum temperature (ERA5). SKILL_START_YEAR and SKILL_END_YEAR define the year range.

**Pearson correlation, r**, measures whether forecast and observed values vary together across years: +1 indicates a perfect positive linear relationship, 0 no linear relationship, and −1 a perfect negative relationship. The plots report the number of matched years, **n**, and use only years with both values available.

The dashed **one-to-one line** shows perfect agreement in absolute values. A high correlation can coexist with a substantial bias. These are raw-value correlations: shared trends, particularly warming, can contribute to them. Correlation alone does not assess ensemble reliability, probability calibration or improvement over a climatological forecast. This is an initial retrospective skill diagnostic, not a complete operational verification.

</details>

In [ ]:
# Settings — Section 7C
SKILL_START_YEAR = 1984
SKILL_END_YEAR = 2024

In [ ]:
forecast, comparison = load_forecasts(COUNTRY, df)
if SKILL_START_YEAR > SKILL_END_YEAR:
    raise ValueError("The start year must be no later than the end year.")
skill_data = comparison.loc[comparison.harvest_year.between(SKILL_START_YEAR, SKILL_END_YEAR)]
fig, axes = plt.subplots(1, 2, figsize=(12, 5.5))
skill_summary = []
for ax, (prefix, observed_column, label, source) in zip(axes, forecast_metrics):
    forecast_column = prefix + "_ensemble_mean"
    pairs = skill_data[["harvest_year", observed_column, forecast_column]].replace(
        [np.inf, -np.inf], np.nan).dropna()
    observed = pairs[observed_column].to_numpy(dtype=float)
    predicted = pairs[forecast_column].to_numpy(dtype=float)
    n = len(pairs)
    if n < 3:
        plt.close(fig)
        raise ValueError("Choose a period with at least three matched harvest years.")
    r = (np.corrcoef(observed, predicted)[0, 1]
         if np.std(observed) > 0 and np.std(predicted) > 0 else np.nan)
    low = min(observed.min(), predicted.min())
    high = max(observed.max(), predicted.max())
    padding = .07 * (high - low) if high > low else 1.0
    limits = (low - padding, high + padding)
    ax.plot(limits, limits, "--", color="grey", lw=1, label="One-to-one line")
    ax.scatter(observed, predicted, s=45, color="#367a9c", alpha=.85,
               edgecolors="white", linewidths=.5)
    ax.set_xlim(limits); ax.set_ylim(limits); ax.set_aspect("equal", adjustable="box")
    ax.set_xlabel(f"Observed ({source})\n{label}")
    ax.set_ylabel(f"Forecast ensemble mean\n{label}")
    correlation_text = f"Pearson r = {r:.2f}" if np.isfinite(r) else "Pearson r undefined (constant values)"
    ax.set_title(f"{correlation_text} | n = {n} years", fontsize=11)
    ax.legend(loc="upper left", fontsize=8); ax.grid(alpha=.2)
    skill_summary.append({"Metric": label, "Observed source": source,
                          "First harvest year": int(pairs.harvest_year.min()),
                          "Last harvest year": int(pairs.harvest_year.max()),
                          "Matched years": n, "Pearson r": r})
fig.suptitle(f"{COUNTRY}: lead-zero ensemble-mean forecast skill | {SKILL_START_YEAR}–{SKILL_END_YEAR}")
plt.tight_layout(); plt.show()

In [ ]:
skill_table = pd.DataFrame(skill_summary)
for i, (prefix, observed_column, label, source) in enumerate(forecast_metrics):
    pairs = skill_data[[observed_column, prefix + "_ensemble_mean"]].dropna()
    skill_table.loc[i, "Average bias (forecast − observed)"] = (pairs.iloc[:, 1] - pairs.iloc[:, 0]).mean()
skill_table["Strength"] = skill_table["Pearson r"].map(strength)
results_table(skill_table, f"{COUNTRY}: forecast skill, {SKILL_START_YEAR}–{SKILL_END_YEAR}",
              formats={"Pearson r": "{:.2f}", "Average bias (forecast − observed)": "{:+.1f}"},
              gradient={"Pearson r": ("Greens", 0, 1)})
key_result("Key result: forecast skill", [
    f"<b>{r['Metric']}</b>: r = <b>{r['Pearson r']:.2f}</b> ({r['Strength']}) over {r['Matched years']} years, "
    f"average bias {r['Average bias (forecast − observed)']:+.1f}." + (" Few years: treat with caution." if r['Matched years'] < 15 else "")
    for _, r in skill_table.iterrows()] + [
    "A high r with a large bias means the forecast rises and falls with reality but is off in absolute terms. "
    "Some of the temperature r can come from warming over the decades, not from season-to-season skill."])

> 💬 **Questions to discuss — Task 7.1 and 7.3**
>
> 1. Which variable has the stronger correlation? Do the plots also show a systematic wet, dry, warm or cool bias?
> 2. Why does a high correlation not necessarily mean a forecast is accurate in absolute terms?
> 3. What other checks would be needed before using these forecasts for agricultural advice?

### 8. What we learned

The box below summarises the main numbers for your country, using the default settings
(1991–2020 reference, OND El Niño index for southern Africa, MAM for Nigeria). Run it after choosing the country in Section 1.

In [ ]:
d, _ = load_country(COUNTRY)
fc, comp = load_forecasts(COUNTRY, d)
trend = np.polyfit(d.harvest_year, d.yield_t_ha, 1)[0] * 10
r_rain = d[["seasonal_rainfall_mm", "local_yield_rank_5"]].corr().iloc[0, 1]
r_temp = d[["mean_tmax_c", "local_yield_rank_5"]].corr().iloc[0, 1]


def tercile_gap(values, ranks, reference):
    """Mean local rank in the upper tercile minus the lower tercile (cut-offs from the reference years)."""
    lower, upper = values[reference].quantile([1/3, 2/3])
    return ranks[values > upper].mean() - ranks[values < lower].mean()


m = comp.merge(d[["harvest_year", "local_yield_rank_5"]], on="harvest_year").dropna(
    subset=["seasonal_rainfall_mm", "rainfall_mm_ensemble_mean", "local_yield_rank_5"])
ref = m.harvest_year.between(1991, 2020)
gap_obs = tercile_gap(m.seasonal_rainfall_mm, m.local_yield_rank_5, ref)
gap_fc = tercile_gap(m.rainfall_mm_ensemble_mean, m.local_yield_rank_5, ref)

oni = pd.read_csv(DATA_DIR / "oni_three_month.csv")
season, offset = ("MAM", 0) if COUNTRY == "Nigeria" else ("OND", -1)
e = d.assign(YR=d.harvest_year + offset).merge(oni[oni.SEAS == season][["YR", "ANOM"]], on="YR", how="left")
rank_nino = e.loc[e.ANOM >= .5, "local_yield_rank_5"].mean()
rank_nina = e.loc[e.ANOM <= -.5, "local_yield_rank_5"].mean()
skill_rain = comp[["seasonal_rainfall_mm", "rainfall_mm_ensemble_mean"]].dropna().corr().iloc[0, 1]
skill_temp = comp[["mean_tmax_c", "mean_tmax_c_ensemble_mean"]].dropna().corr().iloc[0, 1]

learn = pd.DataFrame([
    ("2", "Yield trend", f"{trend:+.2f} t/ha per decade", "Why local ranks help compare years"),
    ("2", "Rainfall vs local rank", f"r = {r_rain:+.2f}", strength(r_rain)),
    ("2", "Max temperature vs local rank", f"r = {r_temp:+.2f}", strength(r_temp)),
    ("3", "Rank gap, wet minus dry years: observed", f"{gap_obs:+.1f}", "How clearly observed rain separates harvests"),
    ("3", "Rank gap, wet minus dry years: forecast", f"{gap_fc:+.1f}", "How clearly the forecast separates harvests"),
    ("6", "Mean rank: El Niño vs La Niña", f"{num(rank_nino, 1)} vs {num(rank_nina, 1)}", f"{season} ONI, ±0.5 °C"),
    ("7", "Forecast skill: rainfall", f"r = {skill_rain:.2f}", strength(skill_rain)),
    ("7", "Forecast skill: max temperature", f"r = {skill_temp:.2f}", strength(skill_temp)),
], columns=["Section", "Measure", COUNTRY, "What it tells us"])
results_table(learn, f"{COUNTRY}: the main numbers from this practical (default settings, harvests 1984–2024)")
if np.sign(gap_fc) == np.sign(gap_obs) and gap_obs != 0:
    forecast_line = (f"Grouping years by <b>observed</b> rainfall gives a rank gap of {gap_obs:+.1f}; by the <b>forecast</b>, "
                     f"{gap_fc:+.1f}: about <b>{100 * gap_fc / gap_obs:.0f} %</b> of the observed gap, in the same direction.")
else:
    forecast_line = (f"Grouping years by <b>observed</b> rainfall gives a rank gap of {gap_obs:+.1f}; by the <b>forecast</b>, "
                     f"{gap_fc:+.1f}: the forecast groups point the other way, so they give no useful rainfall signal here.")
key_result(f"What we learned for {COUNTRY} (default settings, harvests 1984–2024)", [
    f"Yield {'rose' if trend > 0 else 'fell'} by {abs(trend):.2f} t/ha per decade, so <b>local ranks</b> help compare years fairly.",
    f"Seasonal rainfall and local rank: r = {r_rain:+.2f} (<b>{strength(r_rain)}</b>); "
    f"temperature and local rank: r = {r_temp:+.2f} (<b>{strength(r_temp)}</b>).",
    forecast_line,
    f"El Niño years averaged rank {num(rank_nino, 1)} and La Niña years {num(rank_nina, 1)}: a difference in average outcomes, not a certainty.",
    f"Seasonal forecast skill over {len(comp)} years: rainfall r = {skill_rain:.2f}, temperature r = {skill_temp:.2f}."], colour="#2e8b57")

#### Key learnings

| Section | Learning |
|---|---|
| 2 · Weather and yield | National yields have long-term trends. **Local ranks** take most of the trend out, so poor years can be found in any decade. Weather lines up with part of the year-to-year variation, not all of it: rain timing, pests, prices, inputs and data quality matter too. |
| 3 · Forecast groups | Grouping years by **observed** weather usually separates good and poor harvests more clearly than grouping by the **forecast**. Where the forecast keeps part of the signal, it can support a *risk* message. |
| 4 · Hot days and SPI | Different measures capture different kinds of stress. Some line up with yield better than totals and averages; the result can depend on the threshold chosen. |
| 5 · Within the season | Seasonal totals hide timing. A dry or hot month at the wrong stage can matter more than the season's total. |
| 6 · El Niño and La Niña | In some countries El Niño and La Niña years differ in average rainfall and yield, but each group contains good and poor years. |
| 7 · Forecast quality | **Correlation** and **bias** are different things. A forecast can follow the ups and downs well and still be too wet or too hot on average. |

> ✅ **The main message:** in the historical record, poor harvests were more common in some kinds of season. Where a
> forecast picks up part of that pattern, it can support a careful **risk** message such as *"a drier than normal season
> is likely, which raises the risk of a poor harvest"*.
>
> ⚠️ **Too strong a conclusion:** *"the forecast says dry, so the harvest will fail"*. These are national averages
> over about 40 years with small groups, and the composites are not calibrated probabilities of low yield.

### 9. Your country brief (group output)

> 🎯 **Objective:** turn what you found into your own interpretation and advice, and report it back to the room.
>
> **This is the part that counts.** The notebook gives you the evidence; the brief asks for reasoning the notebook
> cannot do for you: what happened in your country, which crop stage was hit, who the national average hides,
> and what a farmer should actually do.
>
> **Before you start:** finish Sections 1–8. The questions draw on Sections 2 (year-by-year table), 4 (hot days),
> 5 (one bad season) and 7 (forecast skill).
>
> **Task sheet:** Group output · everyone

**Questions 1–4 and 6 each have three parts:**

| Part | What to write |
|---|---|
| **Evidence** | a year from the record (1984–2024) **and** at least one other number from this notebook |
| **Interpretation** | your own reasoning or knowledge, something the notebook does *not* show (at least 15 words) |
| **So what** | what it means for farmers or for advice (at least 8 words) |

**Question 5 has its own three parts:** an SMS to farmers (at most 160 characters), two sentences for the Ministry of
Agriculture (at least 15 words), and why the wording differs (at least 8 words).

**The six questions** (for your country, chosen in Section 1). **Every group answers all six** (about 25 minutes).

1. **The year that doesn't fit.** Find a poor harvest (local rank 1 or 2) where rainfall and temperature looked near normal
   (the **Year by year** table at the end of Section 2: rain not more than about 10 % below average and not hotter than
   usual; then look at that year in Section 5). What happened that year that national weather averages cannot show: rainfall timing
   within the season, input subsidies, prices, pests such as fall armyworm (from 2016 in Nigeria, 2016/17 in southern Africa),
   floods, conflict, or how the yield figure was estimated? Say where your explanation comes from.
2. **Which crop stage was hit?** Take a bad season in Section 5 and match its dry or hot months to your country's maize calendar
   (planting, flowering, grain filling). The notebook does not know when farmers plant. Why would stress at that stage matter more?
3. **Who does the national average hide?** Which region or farming system in your country would behave differently, and how would
   that change the advice? What data would show it?
4. **What should a farmer actually do?** If the forecast says below-normal rainfall, and given the forecast skill you found in Section 7,
   what one practical change would you recommend (variety, planting date, inputs, insurance)? What does the farmer lose if the forecast
   is wrong: a dry season that never comes, or a dry season that was missed?
5. **Write the message, twice.** An SMS to farmers (at most 160 characters) and two sentences for the Ministry of Agriculture.
   Both must be honest about uncertainty. Why are they worded differently?
6. **Looking ahead to 2040.** Hot days have risen since the 1980s (Section 4). If that continues, what does it mean for maize in your
   country, what adaptation would you prioritise, and what evidence would you want first?

**Report-back (3 minutes per group, after the break):** present your strongest interpretation (question 1 or 3) and your two messages (question 5).
Each other group asks one challenge question.

**How to save:** type your answers in the cell below and **run it** (Shift+Enter), then **run the cell after it** to check and save
your brief. After any change, run both cells again. The check only looks at the format (every part filled in, a year and a number,
the SMS length); whether your reasoning holds up is for the report-back.

In [ ]:
# ✍️ Type your answers after each colon (answers can run over several lines). Keep the headings and labels.
# Then RUN THIS CELL, and run the next cell to check and save your brief.
GROUP = ""   # your group name, e.g. "Group 3" (each group needs a different name)

BRIEF = """
## 1. The year that doesn't fit
Evidence:
Interpretation:
So what:

## 2. Which crop stage was hit?
Evidence:
Interpretation:
So what:

## 3. Who does the national average hide?
Evidence:
Interpretation:
So what:

## 4. What should a farmer actually do?
Evidence:
Interpretation:
So what:

## 5. Write the message, twice
SMS to farmers:
Ministry of Agriculture:
Why the wording differs:

## 6. Looking ahead to 2040
Evidence:
Interpretation:
So what:
"""

In [ ]:
import hashlib
import re

QUESTIONS = {1: "The year that doesn't fit", 2: "Which crop stage was hit?", 3: "Who does the national average hide?",
             4: "What should a farmer actually do?", 5: "Write the message, twice", 6: "Looking ahead to 2040"}
STANDARD = ["Evidence", "Interpretation", "So what"]
MESSAGES = ["SMS to farmers", "Ministry of Agriculture", "Why the wording differs"]
PARTS = {q: (MESSAGES if q == 5 else STANDARD) for q in QUESTIONS}
MIN_WORDS = {"Interpretation": 15, "So what": 8, "Ministry of Agriculture": 15, "Why the wording differs": 8}
FIRST_YEAR, LAST_YEAR = 1984, 2024
LABEL = re.compile(r"^\s*[-*]?\s*(" + "|".join(re.escape(p) for p in STANDARD + MESSAGES) + r")\s*:", re.I | re.M)
CANONICAL = {p.lower(): p for p in STANDARD + MESSAGES}


def parse_brief(text):
    """Answers by question number and part, plus problems with the headings and labels."""
    text = text.replace("\r\n", "\n")
    answers, problems = {}, []
    sections = re.split(r"^\s*##\s*", text, flags=re.M)
    for section in sections[1:]:
        heading, _, body = section.partition("\n")
        number = re.match(r"\s*(\d+)", heading)
        q = int(number.group(1)) if number else None
        if q not in QUESTIONS:
            problems.append(f"Heading not recognised: '## {heading.strip()}'. Keep the numbered headings 1–6.")
            continue
        if q in answers:
            problems.append(f"Question {q} appears twice: only the first is checked.")
            continue
        found, marks = {}, list(LABEL.finditer(body))
        for i, m in enumerate(marks):
            part = CANONICAL[m.group(1).lower()]
            end = marks[i + 1].start() if i + 1 < len(marks) else len(body)
            if part in found:
                problems.append(f"Question {q}: '{part}' is written twice; only the first is checked.")
            elif part not in PARTS[q]:
                problems.append(f"Question {q}: '{part}' does not belong to this question.")
            else:
                found[part] = body[m.end():end].strip()
        answers[q] = found
    return answers, problems


def check_part(part, answer):
    if answer is None:
        return "missing (heading or label deleted)"
    if not answer:
        return "missing"
    words = len(answer.split())
    if part == "Evidence":
        years = [int(y) for y in re.findall(r"(?<!\d)(1[89]\d\d|20\d\d)(?!\d)", answer)]
        if not any(FIRST_YEAR <= y <= LAST_YEAR for y in years):
            return f"add a year from the record ({FIRST_YEAR}–{LAST_YEAR})"
        other = re.findall(r"\d+(?:\.\d+)?", re.sub(r"(?<!\d)(1[89]\d\d|20\d\d)(?!\d)", " ", answer))
        if not other:
            return "add a number as well as the year (rainfall, rank, r, days…)"
    if part == "SMS to farmers" and len(answer) > 160:
        return f"too long: {len(answer)} characters (max 160)"
    if part == "Ministry of Agriculture" and len([s for s in re.split(r"[.!?]+(?:\s+|$)", answer) if s.strip()]) < 2:
        return "write two sentences"
    if words < MIN_WORDS.get(part, 1):
        return f"too short: {words} words (aim for {MIN_WORDS[part]}+)"
    return "ok"


if "BRIEF" not in globals():
    raise RuntimeError("Run the answer cell above first (click it and press Shift+Enter), then run this cell.")

answers, problems = parse_brief(BRIEF)
rows = []
for q, title in QUESTIONS.items():
    given = answers.get(q, {})
    for part in PARTS[q]:
        rows.append({"Question": f"{q}. {title}", "Part": part, "Status": check_part(part, given.get(part))})
status = pd.DataFrame(rows)
done = int(status.Status.eq("ok").sum())
results_table(status, f"{COUNTRY} country brief: {done} of {len(status)} parts pass the format check",
              colours={"Status": {"ok": "#d4e8d4", "missing": "#f6d5c9", "missing (heading or label deleted)": "#f6d5c9"}})

# file name: one per group; a name that loses characters when made safe gets a short code so groups cannot collide
name = GROUP.strip()
safe = re.sub(r"[^A-Za-z0-9_-]+", "_", name).strip("_")
if not name:
    safe = "unnamed_group"
elif safe != name:
    safe = (safe or "group") + "_" + hashlib.md5(name.encode("utf-8")).hexdigest()[:6]
path = Path(f"country_brief_{COUNTRY}_{safe}.md").resolve()
action = "Updated" if path.exists() else "Saved"
document = f"# Country brief: {COUNTRY}\n\nGroup: {name or '(not given)'}\n\n" + BRIEF.replace("\r\n", "\n").strip() + "\n"
path.write_text(document, encoding="utf-8")

todo = status.loc[status.Status.ne("ok")]
lines = [f"{action} <b>{path}</b>. Hand this file in, or use it for your 3-minute report-back."]
if not name:
    lines.append("⚠️ Set <b>GROUP</b> in the answer cell so your brief is not mixed up with another group's, then run both cells again.")
lines += [f"⚠️ {p}" for p in problems]
lines.append("Format checks passed for all six questions. Whether the reasoning holds up is for the report-back."
             if todo.empty and not problems else
             "Fix the ⚠️ points above, then run both cells again." if todo.empty else
             f"Still to do: <b>{len(todo)}</b> part(s), shown in the table above. Edit the answer cell, then run both cells again.")
key_result("Your brief", lines, colour="#2e8b57" if todo.empty and not problems and name else "#c77d1a")